# 6. Витрина итоговых скоров

Финальная модель скорит всех клиентов `clients`, в том числе 200 123 клиента вне эксперимента: для них витрина
и отвечает на вопрос «кому слать». Признаки - тот же запрос `features` из `sql/features.sql`, витрина и запросы
поверх неё - `sql/scores.sql`. Test здесь не оценивается: метрики не считаются, только скоры.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config, db, evaluate_test, models, scores, stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Признаки для всех клиентов

Логика признаков не смотрит на target, поэтому её можно применить к клиентам вне эксперимента. Сначала - главное
условие отбора в датасет: все клиенты покупали хотя бы раз в окне с 2019-02-24 по 2019-03-18 (этап 1).

In [2]:
features_all = pd.read_parquet(config.FEATURES_ALL_PARQUET)
features_exp = pd.read_parquet(config.FEATURES_PARQUET)
outside = ~features_all.client_id.isin(features_exp.client_id)
print(f"клиентов в витрине: {len(features_all)}, уникальных: {features_all.client_id.nunique()}, вне эксперимента: {int(outside.sum())}")
db.run(db.connect(), "quality.sql", "activity_window")

клиентов в витрине: 400162, уникальных: 400162, вне эксперимента: 200123


,in_experiment,n_clients,earliest_last_purchase,latest_last_purchase
0,True,200039,2019-02-24,2019-03-18
1,False,200123,2019-02-25,2019-03-18


In [3]:
same = features_all.set_index("client_id").loc[features_exp.client_id].reset_index()
num = same.columns.drop(["client_id", "gender"])
max_diff = (same[num] - features_exp[num]).abs().max().max()
n_diff = int(((same[num] != features_exp[num]) & ~(same[num].isna() & features_exp[num].isna())).any(axis=1).sum())
print(f"клиенты эксперимента: признаки совпадают с витриной этапа 3 с точностью {max_diff:.1e}, "
      f"отличия в последних битах у {n_diff} клиентов; пол совпадает: {(same.gender == features_exp.gender).all()}")

клиенты эксперимента: признаки совпадают с витриной этапа 3 с точностью 1.5e-11, отличия в последних битах у 179 клиентов; пол совпадает: True


Для клиентов эксперимента признаки совпадают с витриной этапа 3, кроме последних битов float у части сумм:
DuckDB складывает параллельно, и порядок сложения от запуска к запуску разный. На скоры это не влияет - ниже они
совпадают с сохранёнными точно. Модели обучались на витрине этапа 3, она не пересобиралась.

## Сборка витрины

`python -m src.scores`: скоры финальной модели (`uplift_score`) и модели отклика (`p_t`) для всех клиентов,
таблица долей рассылки по r, выбранных на valid (та же процедура, что в экономике этапа 5), и сама витрина
`data/scores.parquet`. Флаг `send` в сохранённой витрине - для r = 0,05 (`R_DEFAULT` в `config.py`).

In [4]:
scores.build()
con = db.connect()
con.execute(f"create or replace temp view scores as select * from read_parquet('{config.SCORES_MART_PARQUET}')")
db.run(con, "scores.sql", "mart_checks").T

,0
n_rows,400162
n_clients,400162
n_clients_table,400162
n_experiment,200039
n_outside,200123
n_train,120023
n_valid,40008
n_test,40008
n_null_scores,0
n_distinct_uplift_scores,184729


In [5]:
db.run(con, "scores.sql", "decile_sizes")

,in_experiment,uplift_decile,n_clients,min_score,max_score
0,False,1,20013,0.064763,0.332004
1,False,2,20013,0.050143,0.064762
2,False,3,20013,0.041269,0.050143
3,False,4,20012,0.034511,0.041269
4,False,5,20012,0.028791,0.034511
5,False,6,20012,0.023871,0.028791
6,False,7,20012,0.018793,0.023870
7,False,8,20012,0.012558,0.018791
8,False,9,20012,0.005051,0.012558
9,False,10,20012,-0.063288,0.005047


400 162 строки - ровно `clients`, `client_id` уникален, пустых скоров нет. Децили по 20 012-20 013 клиентов вне
эксперимента и по 20 003-20 004 в эксперименте.

## Равные скоры и детерминированность

Разных значений `uplift_score` 184 729, но у большинства клиентов скор не уникален:

In [6]:
db.run(con, "scores.sql", "tied_scores")

,in_experiment,n_clients,share_tied
0,True,200039,0.571893
1,False,200123,0.573692


У 57% клиентов `uplift_score` совпадает хотя бы с одним другим: финальная модель - 16 деревьев по 15 листьев,
и многие клиенты попадают в одни и те же листья. Граница дециля может пройти внутри группы с одинаковым скором
(у 2-го и 3-го дециля вне эксперимента граница 0,050143 с обеих сторон). Поэтому во всех оконных функциях порядок
задан полностью: `order by uplift_score desc, client_id`, для `p_t` - `p_t desc, client_id`. Проверка: витрина,
собранная двумя независимыми запусками на 8 потоках и на одном, даёт одинаковые децили и флаг `send`.

In [7]:
def build_mart(threads):
    c = db.connect()
    c.execute(f"set threads = {threads}")
    return c.execute(db.queries("scores.sql")["scores"].rstrip(";")).df().set_index("client_id")


run_a, run_b = build_mart(8), build_mart(1)
for col in ["uplift_decile", "p_t_decile", "uplift_pct_rank", "send"]:
    print(col, "совпадает:", run_a[col].equals(run_b[col]))

uplift_decile совпадает: True
p_t_decile совпадает: True
uplift_pct_rank совпадает: True
send совпадает: True


## Совпадение скоров с artifacts

Для valid сохранены скоры этапа 4 (`artifacts/valid_scores.parquet`, seed 42). Скоры test скрипт оценки
не сохранял, поэтому для test сверяю витрину со скорами тех же моделей по витрине признаков этапа 3.

In [8]:
mart = pd.read_parquet(config.SCORES_MART_PARQUET).set_index("client_id")
saved = pd.read_parquet(models.ARTIFACTS / "valid_scores.parquet").query(f"seed == {config.SEED}")
saved = saved.pivot(index="client_id", columns="model", values="score")
print("valid, uplift_score = class_transformation:",
      np.array_equal(mart.loc[saved.index, "uplift_score"].to_numpy(), saved["class_transformation"].to_numpy()))
print("valid, p_t = response:", np.array_equal(mart.loc[saved.index, "p_t"].to_numpy(), saved["response"].to_numpy()))
X_test, _, _ = models.load("test")
s_test = evaluate_test.all_scores(evaluate_test.load_models(), X_test)
print("test, uplift_score:", np.array_equal(mart.loc[X_test.index, "uplift_score"].to_numpy(), s_test[evaluate_test.FINAL]))
print("test, p_t:", np.array_equal(mart.loc[X_test.index, "p_t"].to_numpy(), s_test["response"]))

valid, uplift_score = class_transformation: True
valid, p_t = response: True


test, uplift_score: True
test, p_t: True


## Перенос на клиентов вне эксперимента

Модель обучена на клиентах эксперимента. Применять её к остальным можно, если они устроены так же. Проверяю
две вещи: SMD каждого признака витрины между клиентами эксперимента и вне его и долю клиентов вне эксперимента выше
границ децилей скора, посчитанных в эксперименте.

In [9]:
pop = db.run(con, "features.sql", "population_moments")
pop["smd"] = stats.smd(pop.mean_experiment, pop.mean_outside, pop.var_experiment, pop.var_outside)
gender = db.run(con, "features.sql", "population_gender").set_index("in_experiment")
for g in ["gender_f", "gender_m", "gender_u"]:
    p1, p0 = gender.loc[1, g], gender.loc[0, g]
    pop.loc[len(pop)] = {"feature": g, "mean_experiment": p1, "mean_outside": p0,
                         "var_experiment": p1 * (1 - p1), "var_outside": p0 * (1 - p0),
                         "smd": stats.smd(p1, p0, p1 * (1 - p1), p0 * (1 - p0))}
print(f"признаков: {len(pop)}, max |SMD| = {pop.smd.abs().max():.4f} ({pop.loc[pop.smd.abs().idxmax(), 'feature']})")
with pd.option_context("display.max_rows", 60):
    display(pop.reindex(pop.smd.abs().sort_values(ascending=False).index)[
        ["feature", "mean_experiment", "mean_outside", "smd"]].round(4))

признаков: 48, max |SMD| = 0.0066 (exp_spent_14d)


,feature,mean_experiment,mean_outside,smd
9,exp_spent_14d,0.9434,0.9079,0.0066
43,share_spend_7d,0.0647,0.0660,-0.0063
4,days_since_first,97.2689,97.0994,0.0063
10,share_spend_all,0.0718,0.0724,-0.0060
12,n_trn_7d,1.3755,1.3667,0.0051
32,exp_spent_all,6.4543,6.3838,0.0043
34,n_stores_7d,0.8008,0.7975,0.0043
3,days_since_last,5.8289,5.8496,-0.0037
31,reg_spent_7d,4.5044,4.6061,-0.0036
30,avg_check_30d,492.4177,493.8899,-0.0035


In [10]:
transfer = db.run(con, "scores.sql", "decile_transfer")
dev = (transfer.share_outside_above - transfer.expected_share).abs().max()
print(f"max |доля выше границы - ожидаемая| = {dev * 100:.2f} п.п.")
transfer.round(4)

max |доля выше границы - ожидаемая| = 0.13 п.п.


,uplift_decile,lower_bound,expected_share,share_outside_above
0,1,0.0648,0.1,0.1000
1,2,0.0500,0.2,0.2011
2,3,0.0413,0.3,0.3002
3,4,0.0345,0.4,0.4006
4,5,0.0288,0.5,0.4997
5,6,0.0239,0.6,0.5996
6,7,0.0187,0.7,0.7013
7,8,0.0126,0.8,0.7996
8,9,0.0050,0.9,0.9004
9,10,-0.0640,1.0,1.0000


Распределения совпадают: |SMD| не больше 0,0066 по всем 48 признакам (45 числовых и три доли пола), а выше границ
децилей эксперимента лежит та же доля клиентов вне эксперимента, отклонение не больше 0,13 п.п. Похоже, что
клиентов `clients` делили на эксперимент и тест хакатона случайно (предположение). Признаки и скоры переносятся,
но эффект вне эксперимента не измерен: это экстраполяция модели на похожих клиентов.

## Децили скора вне эксперимента

In [11]:
db.run(con, "scores.sql", "decile_summary").round(3)

,uplift_decile,n_clients,avg_uplift_score,avg_p_t,days_since_last,n_trn_all,sum_all,card_age_days,share_spend_all,age
0,1,20013,0.094,0.599,5.0,10.0,2824.950,276.0,0.149,58.0
1,2,20013,0.057,0.592,5.0,11.0,4231.290,358.0,0.080,57.0
2,3,20013,0.045,0.608,4.0,13.0,5246.500,452.0,0.072,55.0
3,4,20012,0.038,0.623,4.0,14.0,5826.000,483.0,0.069,52.0
4,5,20012,0.032,0.630,4.0,15.0,6314.215,508.0,0.068,48.0
5,6,20012,0.026,0.635,4.0,16.0,6708.300,529.0,0.063,45.0
6,7,20012,0.021,0.639,4.0,17.0,7293.510,541.0,0.060,41.0
7,8,20012,0.016,0.642,4.0,18.0,7375.025,538.0,0.056,37.0
8,9,20012,0.009,0.659,4.0,21.0,9170.970,552.0,0.053,33.0
9,10,20012,-0.004,0.734,2.0,29.0,12239.500,517.0,0.055,31.0


Это описание, а не объяснение. Чем выше дециль uplift-скора, тем меньше чеков и сумма за историю, моложе карта,
старше клиент и выше доля чеков со списанием баллов. В нижнем дециле - самые активные клиенты с самой высокой
вероятностью покупки (средний p_t 0,73), им сообщение по оценке модели почти ничего не добавляет.

## Кому слать при заданном r

r - допущение, задаётся переменной запроса. Доля k для r берётся из выбора на valid. Для нескольких r:

In [12]:
rows = []
for r in [0.0, 0.0325, 0.05, 0.1, 0.2, 0.3]:
    con.execute(f"set variable r = {r}")
    db.create_view(con, "scores.sql", "scores")
    rows.append(db.run(con, "scores.sql", "send_summary"))
pd.concat(rows).set_index("r")

,k_from_valid,n_send,n_outside,share_send
r,,,,
0.0000,0.97,194119,200123,0.969998
0.0325,0.50,100061,200123,0.499998
0.0500,0.14,28017,200123,0.139999
0.1000,0.04,8004,200123,0.039995
0.2000,0.01,2001,200123,0.009999
0.3000,0.00,0,200123,0.000000


In [13]:
con.execute(f"set variable r = {config.R_DEFAULT}")
db.create_view(con, "scores.sql", "scores")
send = db.run(con, "scores.sql", "send_list")
print(f"r = {config.R_DEFAULT}: в списке {len(send)} клиентов")
send.head()

r = 0.05: в списке 28017 клиентов


,client_id,uplift_score,uplift_decile,p_t
0,7c5f52a8de,0.332004,1,0.819502
1,808131fc73,0.328805,1,0.710068
2,0fe6dacaaf,0.326713,1,0.686775
3,e8678f6b89,0.326713,1,0.695668
4,fb32901946,0.325240,1,0.759038


## Пересечение топов модели отклика и uplift-модели

In [14]:
db.run(con, "scores.sql", "top_overlap").round(4)

,overlap_top10,overlap_top30
0,0.0394,0.2114


Среди клиентов вне эксперимента в топ-10% по uplift-модели только 3,9% клиентов из топ-10% модели отклика (при
случайном выборе было бы 10%), в топ-30% - 21,1% (было бы 30%). На valid пересечение топ-30% было 21,9%:
модели отбирают почти разных клиентов и вне эксперимента.

## Итоги

- Витрина `data/scores.parquet`: 400 162 клиента, `uplift_score`, `p_t`, перцентиль и дециль скора отдельно для
  эксперимента и вне его, флаг `send` для заданного r.
- Скоры в витрине совпадают с сохранёнными скорами valid и со скорами test по витрине этапа 3 точно. У 57% клиентов
  скор не уникален, порядок в оконных функциях задан с `client_id`, два запуска дают одинаковый `send`.
- Клиенты вне эксперимента по признакам не отличаются от клиентов эксперимента: |SMD| не больше 0,0066, доли
  выше границ децилей - как в эксперименте.
- При r = 0,05 сообщение получают 14% клиентов вне эксперимента - 28 017 человек; при r = 0,0325 - половина,
  при r = 0,3 - никто. Доля взята из выбора на valid, ожидаемая прибыль при ней оценена на test (этап 5).